# 23 - Dataset Replay Producer

## Purpose

This notebook simulates a real-time transaction stream by replaying transactions from the prepared replay dataset into Apache Kafka.

Unlike the previous producer, this notebook does **not** generate synthetic transactions. Instead, it streams realistic transactions from the replay dataset to simulate a live payment processing system.

---

## Input

- `data/replay/creditcard_replay.csv`

---

## Output

Kafka Topic:

- `fraud_transactions`

---

## Features

- Dataset replay
- Configurable streaming modes
- Configurable replay speed
- Event timestamp generation
- Event metadata
- Logging
- Progress monitoring
- Graceful shutdown

---

## Streaming Modes

- Fixed Rate
- Random Rate
- Burst Mode
- Peak Hour Simulation

---

## Architecture

Replay Dataset

↓

Kafka Producer

↓

Kafka Topic

↓

Spark Streaming

↓

Fraud Prediction

↓

PostgreSQL

↓

Power BI Dashboard

# Section 1 — Imports

Import all required libraries for:

- Kafka communication
- Dataset handling
- Logging
- Event timestamp generation
- Performance monitoring

In [ ]:
# ============================================
# Section 1 - Imports
# ============================================

import os
import json
import uuid
import time
import random
import signal
import logging

from datetime import datetime, timedelta, timezone

import pandas as pd

from kafka import KafkaProducer

print("Libraries imported successfully.")

# Section 2 — Configuration

Centralize all configuration variables.

Changing replay speed, Kafka topic, or streaming mode should only require editing this section.

In [ ]:
# ============================================
# Section 2 - Configuration
# ============================================

# Replay Dataset
DATASET_PATH = "../data/replay/creditcard_replay.csv"

# Kafka
BOOTSTRAP_SERVERS = "localhost:9092"

TOPIC_NAME = "fraud_transactions"

# Streaming
STREAM_MODE = "fixed"

STREAM_DELAY = 0.10

MIN_DELAY = 0.05

MAX_DELAY = 1.50

BURST_SIZE = 100

BURST_PAUSE = 3

SHUFFLE_DATA = False

LOOP_STREAM = False

MAX_TRANSACTIONS = None

# Logging
LOG_DIRECTORY = "../logs"

LOG_FILE = os.path.join(
    LOG_DIRECTORY,
    "dataset_replay.log"
)

# Producer Information
PRODUCER_VERSION = "2.0"

PRODUCER_NAME = "Dataset Replay Producer"

os.makedirs(LOG_DIRECTORY, exist_ok=True)

print("Configuration loaded successfully.")

# Section 2.5 — Configuration Validation

Validate all configuration values before starting the replay producer.

This prevents common configuration errors such as:

- Invalid streaming mode
- Negative delays
- Invalid burst configuration
- Missing Kafka topic
- Missing bootstrap server

In [ ]:
# ============================================
# Section 2.5 - Configuration Validation
# ============================================
# ============================================
# Streaming Mode Constants
# ============================================

FIXED_MODE = "fixed"

RANDOM_MODE = "random"

BURST_MODE = "burst"

PEAK_HOUR_MODE = "peak_hour"
print("=" * 70)
print("Configuration Validation")
print("=" * 70)

SUPPORTED_STREAM_MODES = [

    FIXED_MODE,

    RANDOM_MODE,

    BURST_MODE,

    PEAK_HOUR_MODE

]

# --------------------------------------------------
# Stream Mode
# --------------------------------------------------

if STREAM_MODE not in SUPPORTED_STREAM_MODES:

    raise ValueError(

        f"Unsupported STREAM_MODE: {STREAM_MODE}\n"

        f"Supported Modes: {SUPPORTED_STREAM_MODES}"

    )

print(f"✓ Stream Mode           : {STREAM_MODE}")

# --------------------------------------------------
# Dataset Path
# --------------------------------------------------

if not DATASET_PATH:

    raise ValueError(

        "DATASET_PATH cannot be empty."

    )

print(f"✓ Replay Dataset        : {DATASET_PATH}")

# --------------------------------------------------
# Kafka Configuration
# --------------------------------------------------

if not BOOTSTRAP_SERVERS:

    raise ValueError(

        "BOOTSTRAP_SERVERS cannot be empty."

    )

if not TOPIC_NAME:

    raise ValueError(

        "TOPIC_NAME cannot be empty."

    )

print(f"✓ Kafka Server          : {BOOTSTRAP_SERVERS}")

print(f"✓ Kafka Topic           : {TOPIC_NAME}")

# --------------------------------------------------
# Fixed Mode Validation
# --------------------------------------------------

if STREAM_MODE == FIXED_MODE:

    if STREAM_DELAY <= 0:

        raise ValueError(

            "STREAM_DELAY must be greater than zero."

        )

    print(f"✓ Stream Delay          : {STREAM_DELAY:.2f} sec")

# --------------------------------------------------
# Random Mode Validation
# --------------------------------------------------

elif STREAM_MODE == RANDOM_MODE:

    if MIN_DELAY <= 0:

        raise ValueError(

            "MIN_DELAY must be greater than zero."

        )

    if MAX_DELAY <= 0:

        raise ValueError(

            "MAX_DELAY must be greater than zero."

        )

    if MIN_DELAY >= MAX_DELAY:

        raise ValueError(

            "MIN_DELAY must be smaller than MAX_DELAY."

        )

    print(f"✓ Random Delay          : {MIN_DELAY:.2f} - {MAX_DELAY:.2f} sec")

# --------------------------------------------------
# Burst Mode Validation
# --------------------------------------------------

elif STREAM_MODE == BURST_MODE:

    if BURST_SIZE <= 0:

        raise ValueError(

            "BURST_SIZE must be greater than zero."

        )

    if BURST_PAUSE < 0:

        raise ValueError(

            "BURST_PAUSE cannot be negative."

        )

    print(f"✓ Burst Size            : {BURST_SIZE}")

    print(f"✓ Burst Pause           : {BURST_PAUSE} sec")

# --------------------------------------------------
# Peak Hour Mode Validation
# --------------------------------------------------

elif STREAM_MODE == PEAK_HOUR_MODE:

    print("✓ Peak Hour Simulation  : Enabled")

# --------------------------------------------------
# Replay Configuration
# --------------------------------------------------

print(f"✓ Shuffle Dataset       : {SHUFFLE_DATA}")

print(f"✓ Loop Replay           : {LOOP_STREAM}")

if MAX_TRANSACTIONS is None:

    print("✓ Maximum Transactions  : ALL")

else:

    if MAX_TRANSACTIONS <= 0:

        raise ValueError(

            "MAX_TRANSACTIONS must be greater than zero."

        )

    print(f"✓ Maximum Transactions  : {MAX_TRANSACTIONS}")

# --------------------------------------------------
# Producer Information
# --------------------------------------------------

print(f"✓ Producer Name         : {PRODUCER_NAME}")

print(f"✓ Producer Version      : {PRODUCER_VERSION}")

print("=" * 70)
print("Configuration Validation Passed Successfully")
print("=" * 70)

logging.info("Configuration validation completed successfully.")

# Section 3 — Logging

Initialize the logging system.

Every execution is recorded for monitoring and debugging.

In [ ]:
# ============================================
# Section 3 - Logging
# ============================================

logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    filemode="a"

)

logging.info("=" * 80)
logging.info("Dataset Replay Producer Started")
logging.info("=" * 80)

logging.info(f"Producer : {PRODUCER_NAME}")
logging.info(f"Version  : {PRODUCER_VERSION}")

print("Logging initialized.")

# Section 4 — Runtime Variables

Initialize variables used during streaming.

These values are updated while the producer is running.

In [ ]:
# ============================================
# Section 4 - Runtime Variables
# ============================================

start_time = None

transactions_sent = 0

stream_running = True

current_batch = 1

print("Runtime variables initialized.")

# Section 5 — Graceful Shutdown

Allow the producer to stop safely when the user presses **Ctrl + C**.

In [ ]:
# ============================================
# Section 5 - Graceful Shutdown
# ============================================

def shutdown_handler(signum, frame):

    global stream_running

    stream_running = False

    print("\nStopping Producer...")

    logging.info("Producer stopped by user.")

signal.signal(signal.SIGINT, shutdown_handler)

print("Graceful shutdown registered.")

# Section 6 — Environment Validation

Verify that the replay dataset exists before attempting to stream transactions.

In [ ]:
# ============================================
# Section 6 - Environment Validation
# ============================================

if not os.path.exists(DATASET_PATH):

    raise FileNotFoundError(

        f"Replay dataset not found:\n{DATASET_PATH}"

    )

logging.info("Replay dataset found.")

print("Replay dataset verified.")

# ==========================================================
# Section 6.5 - Load Replay Dataset
# ==========================================================

In [ ]:


print("=" * 70)
print("Loading Replay Dataset")
print("=" * 70)

logging.info("Loading replay dataset...")

df = pd.read_csv(DATASET_PATH)

print(f"Dataset Loaded Successfully")

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

logging.info(
    f"Replay dataset loaded successfully ({len(df):,} rows)"
)

display(df.head())

# Section 7 — Kafka Producer

Create and validate the Kafka producer.

This section:

- Creates the Kafka producer
- Verifies the Kafka broker is reachable
- Configures JSON serialization
- Handles connection failures gracefully

In [ ]:
# ==========================================================
# Section 7 - Create Kafka Producer
# ==========================================================

print("=" * 70)
print("Creating Kafka Producer")
print("=" * 70)

logging.info("Creating Kafka Producer...")

try:

    producer = KafkaProducer(

        bootstrap_servers=BOOTSTRAP_SERVERS,

        value_serializer=lambda value:
            json.dumps(value).encode("utf-8")

    )

    print("✓ Kafka Producer Connected")
    logging.info("Kafka Producer Connected Successfully")

except Exception as e:

    logging.exception("Kafka Producer Connection Failed")

    raise RuntimeError(
        f"""
Failed to connect to Kafka.

Bootstrap Server:
{BOOTSTRAP_SERVERS}

Error:
{e}

Please verify:
1. Kafka is running.
2. Docker containers are running.
3. Port 9092 is available.
"""
    )

# Section 8 — Test Kafka Connection

Send a small test event to verify that Kafka is accepting messages before starting the replay.

In [ ]:
print("\nTesting Kafka Connection...\n")

test_event = {

    "event_type": "connection_test",

    "producer": PRODUCER_NAME,

    "version": PRODUCER_VERSION,

    "timestamp": datetime.now().isoformat()

}

try:

    future = producer.send(

        TOPIC_NAME,

        value=test_event

    )

    metadata = future.get(timeout=10)

    print("✓ Kafka connection successful!")

    print(f"Topic     : {metadata.topic}")

    print(f"Partition : {metadata.partition}")

    print(f"Offset    : {metadata.offset}")

    logging.info("Kafka connection test successful.")

except Exception:

    logging.exception("Kafka connection test failed.")

    raise

# Section 9 — Flush Producer

Flush the producer to ensure that all buffered messages have been delivered.

In [ ]:
# ============================================
# Section 9 - Flush Producer
# ============================================

producer.flush()

print("✓ Producer Flush Successful")

logging.info("Producer flushed.")

# Section 10 — Producer Health Check

Display the current producer configuration and verify readiness for dataset replay.

In [ ]:
# ============================================
# Section 10 - Producer Health Check
# ============================================

print("=" * 70)

print("Kafka Producer Ready")

print("=" * 70)

print(f"Producer Name      : {PRODUCER_NAME}")

print(f"Producer Version   : {PRODUCER_VERSION}")

print(f"Kafka Server       : {BOOTSTRAP_SERVERS}")

print(f"Topic              : {TOPIC_NAME}")

print(f"Streaming Mode     : {STREAM_MODE}")

print("=" * 70)

logging.info("Kafka producer ready.")

# Section 10.5 — Replay Configuration

Configure replay behaviour.

These settings control:

- Replay speed
- Logging frequency
- Total records

In [ ]:
# ==========================================================
# Section 10.5 - Replay Configuration
# ==========================================================

print("=" * 70)
print("Replay Configuration")
print("=" * 70)

# Total number of records to replay
TOTAL_RECORDS = len(df)

# Delay between messages (seconds)
REPLAY_DELAY_SECONDS = 0.1

# Log progress every N records
LOG_INTERVAL = 10

print(f"Total Records        : {TOTAL_RECORDS:,}")
print(f"Replay Delay         : {REPLAY_DELAY_SECONDS} second(s)")
print(f"Logging Interval     : {LOG_INTERVAL}")

logging.info(
    f"""
Replay Configuration

Total Records : {TOTAL_RECORDS}
Replay Delay  : {REPLAY_DELAY_SECONDS}
Log Interval  : {LOG_INTERVAL}
"""
)

# Section 11 — Replay Dataset

Replay every transaction from the prepared replay dataset.

For each transaction:

- Generate a new Transaction ID
- Generate a fresh Event Timestamp
- Publish to Kafka
- Log progress
- Respect the configured replay speed

In [ ]:
# ==========================================================
# Section 11 - Replay Dataset to Kafka
# ==========================================================

print("=" * 70)
print("Starting Dataset Replay")
print("=" * 70)

logging.info("Starting dataset replay.")

records_sent = 0
start_time = time.time()

for index, row in df.iterrows():

    # -----------------------------------------
    # Convert row to dictionary
    # -----------------------------------------

    message = row.to_dict()

    # -----------------------------------------
    # Generate runtime metadata
    # -----------------------------------------

    message["transaction_id"] = str(uuid.uuid4())

    message["event_timestamp"] = (
        datetime.now(timezone.utc)
        .isoformat(timespec="milliseconds")
    )

    # -----------------------------------------
    # Publish message
    # -----------------------------------------

    producer.send(

        TOPIC_NAME,

        value=message

    )

    records_sent += 1

    # -----------------------------------------
    # Progress logging
    # -----------------------------------------

    if records_sent % LOG_INTERVAL == 0:

        print(
            f"Published "
            f"{records_sent:,} / {TOTAL_RECORDS:,} transactions"
        )

        logging.info(
            f"Published {records_sent:,} of {TOTAL_RECORDS:,}"
        )

    # -----------------------------------------
    # Replay speed
    # -----------------------------------------

    time.sleep(REPLAY_DELAY_SECONDS)

# Section 12 — Replay Summary

Display a summary of the completed replay session including:

- Total transactions published
- Replay duration
- Average throughput
- Replay mode

In [ ]:
# ==========================================================
# Section 12 — Replay Summary
# ==========================================================

print("=" * 70)
print("Replay Summary")
print("=" * 70)

end_time = time.time()

elapsed_time = end_time - start_time

throughput = (
    records_sent / elapsed_time
    if elapsed_time > 0
    else 0
)

print(f"Transactions Published : {records_sent:,}")
print(f"Elapsed Time           : {elapsed_time:.2f} seconds")
print(f"Average Throughput     : {throughput:.2f} msg/sec")

logging.info(
    f"""
Replay Summary

Transactions : {records_sent}
Elapsed Time : {elapsed_time:.2f}
Throughput   : {throughput:.2f}
"""
)

# Section 13 — Producer Cleanup

Flush any remaining messages and close the Kafka producer safely.

In [ ]:
# ==========================================================
# Section 13 — Producer Cleanup
# ==========================================================

print("\nClosing Kafka Producer...\n")

try:

    producer.flush()

    producer.close()

    print("Kafka Producer Closed Successfully")

    logging.info("Kafka producer closed successfully.")

except Exception:

    logging.exception("Failed while closing producer.")

    raise

# Section 14 — Final Statistics

Display useful runtime statistics.

In [ ]:
# ==========================================================
# Section 14 — Final Statistics
# ==========================================================

print("=" * 70)
print("Runtime Statistics")
print("=" * 70)

statistics = {

    "Producer": PRODUCER_NAME,

    "Version": PRODUCER_VERSION,

    "Topic": TOPIC_NAME,

    "Records Published": records_sent,

    "Replay Delay (sec)": REPLAY_DELAY_SECONDS,

    "Log Interval": LOG_INTERVAL,

    "Throughput (msg/sec)": round(throughput, 2),

    "Started": datetime.fromtimestamp(start_time),

    "Finished": datetime.fromtimestamp(end_time)

}

for key, value in statistics.items():

    print(f"{key:<25}: {value}")

logging.info(statistics)

# Section 15 — Notebook Summary

Dataset replay completed successfully.

This notebook:

- Loaded the replay dataset
- Validated the dataset
- Connected to Kafka
- Streamed every transaction
- Generated runtime metadata
- Logged progress
- Closed Kafka gracefully

The fraud prediction pipeline is now ready to consume these events.